In [2]:

# ==========================================
# IMPORT LIBRARIES
# ==========================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch


# ==========================================
# 1. API KEYS
# ==========================================

GOOGLE_API_KEY = ""
TAVILY_API_KEY = ""


# ==========================================
# 2. INITIALIZE GEMINI MODEL
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


# ==========================================
# 3. TOOL 1 - SIMPLE QA
# ==========================================

qa_prompt = ChatPromptTemplate.from_template(
    "Answer the following question clearly:\n{question}"
)

qa_chain = qa_prompt | llm

qa_tool = Tool(
    name="Simple_QA",
    func=lambda question: qa_chain.invoke(
        {"question": question}
    ).content,
    description=(
        "Use this tool to answer general factual questions "
        "and explain technical concepts in 20 words."
    )
)


# ==========================================
# 4. TOOL 2 - SUMMARIZER
# ==========================================

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text clearly:\n\n{text}"
)

summary_chain = summary_prompt | llm

summary_tool = Tool(
    name="Summarizer",
    func=lambda text: summary_chain.invoke(
        {"text": text}
    ).content,
    description=(
        "Use this tool to summarize long paragraphs, "
        "documents, and articles in 20words."
    )
)


# ==========================================
# 5. TOOL 3 - TAVILY WEB SEARCH
# ==========================================

search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=TAVILY_API_KEY
)


# ==========================================
# 6. CREATE TOOL LIST
# ==========================================

tools = [
    qa_tool,
    summary_tool,
    search_tool
]


# ==========================================
# 7. CREATE LANGCHAIN AGENT
# ==========================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Choose the most appropriate tool based on the user's request. "
        "Use Simple_QA for factual questions. "
        "Use Summarizer for summarization. "
        "Use tavily search for current web information. "
        "If no tool is necessary, answer directly."
        "Answer the question in 20 words"
    )
)


# ==========================================
# 8. RUN USER QUERIES
# ==========================================

queries = [
    "What is LangGraph in LangChain?",

    "Summarize: Artificial Intelligence helps machines "
    "perform tasks that normally require human intelligence.",

    "What are the latest developments in Generative AI?"
]


for query in queries:

    print("\n" + "=" * 60)
    print("USER QUERY:", query)
    print("=" * 60)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )


    # ======================================
    # 9. DISPLAY TOOL EXECUTION DETAILS
    # ======================================

    print("\n--- TOOL EXECUTION DETAILS ---")

    tool_used = False

    for message in response["messages"]:

        # Display selected tool
        if getattr(message, "tool_calls", None):

            for tool_call in message.tool_calls:

                tool_used = True

                print("\nSelected Tool:", tool_call["name"])
                print("Tool Input:", tool_call["args"])


        # Display tool result
        if getattr(message, "type", None) == "tool":

            print("\nTool Result:")
            print(message.content)


    if not tool_used:

        print("No tool was called. Agent answered directly.")


    # ======================================
    # 10. DISPLAY FINAL RESPONSE
    # ======================================

    print("\n--- FINAL AGENT RESPONSE ---")

    print(response["messages"][-1].content)

    print("\n")



USER QUERY: What is LangGraph in LangChain?

--- TOOL EXECUTION DETAILS ---

Selected Tool: Simple_QA
Tool Input: {'__arg1': 'What is LangGraph in LangChain?'}

Tool Result:
[{'type': 'text', 'text': '**LangGraph** is a library built by the creators of LangChain designed to build **complex, stateful, multi-agent AI applications** using a graph-based structure. \n\nWhile standard LangChain is great for creating linear chains of tasks (Step A → Step B → Step C), **LangGraph allows you to build workflows with loops, memory, and multiple AI agents working together.**\n\n---\n\n### Key Concepts: Why use LangGraph?\n\n#### 1. Cycles and Loops (Instead of linear chains)\nStandard chains are "Directed Acyclic Graphs" (DAGs), meaning information only moves forward. However, real-world AI agents often need to **loop**:\n* *Example:* An agent writes code → tests the code → finds a bug → **loops back** to rewrite the code → tests again until it succeeds. \nLangGraph natively supports these cyclic